# 01 - Exploratory Data Analysis

This notebook starts with a structural review of the IBM Telco customer churn data: its dimensions, values, data types, and missingness.

## Load the raw data

The project loader validates that the expected raw schema is present before returning the dataframe.

In [1]:
import pandas as pd

from churn_ml.data.load_data import load_raw_data

df = load_raw_data()
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Dataset shape: 7,043 rows x 33 columns


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


## Initial assessment

- The raw dataset contains **7,043 customer records** and **33 columns**.
- Pandas initially reads 9 columns as numeric (`int64` or `float64`) and 24 as text-like (`object`).
- `Churn Reason` is the only column with explicit null values: **5,174 (73.5%)**. This is expected for customers who did not churn, but it must not be used as a predictive feature because it is post-outcome information.
- `Total Charges` is currently an `object` column even though it represents an amount; inspect and convert it to numeric during data cleaning.
- `CustomerID` is an identifier, `Count` is a constant field, and `Churn Label` / `Churn Score` are outcome-related fields. These should be reviewed for exclusion before modeling to avoid leakage.

## Dataframe summary

`info()` gives a compact schema and non-null count, while `describe(include='all')` summarizes both numeric and categorical values.

In [2]:
df.info()

summary_statistics = df.describe(include="all").T
summary_statistics

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null   object 
 16  Internet Service   7043 

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
CustomerID,7043,7043,3668-QPYBK,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Count,7043.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0
Country,7043,1,United States,7043,NaN,NaN,NaN,NaN,NaN,NaN,NaN
State,7043,1,California,7043,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,7043,1129,Los Angeles,305,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Zip Code,7043.0,NaN,NaN,NaN,93521.964646,1865.794555,90001.0,92102.0,93552.0,95351.0,96161.0
Lat Long,7043,1652,"34.159534, -116.425984",5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Latitude,7043.0,NaN,NaN,NaN,36.282441,2.455723,32.555828,34.030915,36.391777,38.224869,41.962127
Longitude,7043.0,NaN,NaN,NaN,-119.79888,2.157889,-124.301372,-121.815412,-119.730885,-118.043237,-114.192901
Gender,7043,2,Male,3555,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data types and missing values

The following table is a column-level audit. It reports each pandas data type, number of distinct values, non-null count, and the count and percentage of missing values.

In [3]:
column_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "distinct_values": df.nunique(dropna=True),
    "non_null_count": df.notna().sum(),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
})

column_summary.sort_values(["missing_count", "dtype"], ascending=[False, True])

,dtype,distinct_values,non_null_count,missing_count,missing_pct
Churn Reason,object,20,1869,5174,73.46
Latitude,float64,1652,7043,0,0.00
Longitude,float64,1651,7043,0,0.00
Monthly Charges,float64,1585,7043,0,0.00
Count,int64,1,7043,0,0.00
Zip Code,int64,1652,7043,0,0.00
Tenure Months,int64,73,7043,0,0.00
Churn Value,int64,2,7043,0,0.00
Churn Score,int64,85,7043,0,0.00
CLTV,int64,3438,7043,0,0.00


## Preliminary feature assessment

The prediction target is **`Churn Value`** (`1` = churned; `0` = remained). The candidate predictors below describe the customer's relationship, services, contract, and billing at the time of prediction. `CLTV` is retained for additional analysis but is kept separate from the initial prediction-feature set because it is a predicted value.

To make service categories consistent, treat `No phone service` in `Multiple Lines` as `No`; treat `No internet service` as `No` in each internet add-on field. This normalization is displayed here without changing the raw dataframe; it will be applied in the data-cleaning stage.

`Gender`, `Senior Citizen`, `City`, `State`, `Zip Code`, `Latitude`, and `Longitude` are excluded from prediction to reduce discrimination and fairness-bias risks. They are retained as **fairness-monitoring fields** for later group-level performance and outcome analysis.

In [4]:
target_column = "Churn Value"

prediction_features = [
    "Partner",
    "Dependents",
    "Tenure Months",
    "Phone Service",
    "Multiple Lines",
    "Internet Service",
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies",
    "Contract",
    "Paperless Billing",
    "Payment Method",
    "Monthly Charges",
    "Total Charges",
]

additional_analysis_columns = ["CLTV"]
fairness_monitoring_columns = [
    "Gender",
    "Senior Citizen",
    "City",
    "State",
    "Zip Code",
    "Latitude",
    "Longitude",
]

assert set(prediction_features + additional_analysis_columns + fairness_monitoring_columns + [target_column]).issubset(df.columns)

selection_summary = pd.DataFrame({
    "column": prediction_features + additional_analysis_columns + fairness_monitoring_columns + [target_column],
    "role": (
        ["prediction feature"] * len(prediction_features)
        + ["additional analysis"] * len(additional_analysis_columns)
        + ["fairness monitoring only"] * len(fairness_monitoring_columns)
        + ["prediction target"]
    ),
})
selection_summary

,column,role
0,Partner,prediction feature
1,Dependents,prediction feature
2,Tenure Months,prediction feature
3,Phone Service,prediction feature
4,Multiple Lines,prediction feature
5,Internet Service,prediction feature
6,Online Security,prediction feature
7,Online Backup,prediction feature
8,Device Protection,prediction feature
9,Tech Support,prediction feature


## Prediction dataset

Create an EDA dataframe with only the approved prediction features and the churn target. Service-category normalization remains a later data-cleaning step, so this dataframe preserves the raw values.

In [5]:
prediction_df = df[prediction_features + [target_column]].copy()

# Convert numeric fields that were imported as strings (for example, blank Total Charges values).
numeric_features = ["Tenure Months", "Monthly Charges", "Total Charges"]
prediction_df["Total_Charges_Missing"] = (
    pd.to_numeric(prediction_df["Total Charges"], errors="coerce").isna().astype(int)
)
for column in numeric_features:
    prediction_df[column] = pd.to_numeric(prediction_df[column], errors="coerce")

# Encode only feature columns; retain the target in its original form.
categorical_features = prediction_df[prediction_features].select_dtypes(include=["object", "string", "category"]).columns.tolist()
category_counts = prediction_df[categorical_features].nunique(dropna=True)
binary_features = category_counts[category_counts == 2].index.tolist()
one_hot_features = category_counts[category_counts.between(3, 5)].index.tolist()
unsupported_features = category_counts[~category_counts.isin([2, 3, 4, 5])].index.tolist()
if unsupported_features:
    raise ValueError(f"Unexpected category count for: {unsupported_features}")

for column in binary_features:
    values = set(prediction_df[column].dropna().unique())
    if values != {"No", "Yes"}:
        raise ValueError(f"Binary feature {column!r} must contain only Yes/No values; found {values}")
    prediction_df[column] = prediction_df[column].map({"No": 0, "Yes": 1}).astype("Int64")

prediction_df = pd.get_dummies(
    prediction_df,
    columns=one_hot_features,
    dtype=int,
)

# Preserve the full encoded feature set for XGBoost before linear-model pruning.
xgboost_prediction_df = prediction_df.copy()

print(f"Processed prediction dataset shape: {prediction_df.shape[0]:,} rows x {prediction_df.shape[1]} columns")
print(f"Binary-encoded columns: {len(binary_features)}; one-hot encoded columns: {len(one_hot_features)}")
prediction_df.dtypes

Processed prediction dataset shape: 7,043 rows x 40 columns
Binary-encoded columns: 4; one-hot encoded columns: 10


Partner                                       Int64
Dependents                                    Int64
Tenure Months                                 int64
Phone Service                                 Int64
Paperless Billing                             Int64
Monthly Charges                             float64
Total Charges                               float64
Churn Value                                   int64
Total_Charges_Missing                         int64
Multiple Lines_No                             int64
Multiple Lines_No phone service               int64
Multiple Lines_Yes                            int64
Internet Service_DSL                          int64
Internet Service_Fiber optic                  int64
Internet Service_No                           int64
Online Security_No                            int64
Online Security_No internet service           int64
Online Security_Yes                           int64
Online Backup_No                              int64
Online Backu

## Predictor correlation with churn

This heatmap uses Pearson correlation to show the direction and strength of the linear relationship between each processed predictor and `Churn Value`. An asterisk (`*`) marks correlations whose 95% confidence interval includes zero (not statistically different from zero at the unadjusted 5% level). Correlation highlights associations only; it does not establish causation.

In [6]:
import numpy as np
import plotly.graph_objects as go

predictor_columns = [column for column in prediction_df.columns if column != target_column]
correlation_results = []
for column in predictor_columns:
    paired_values = prediction_df[[column, target_column]].dropna()
    correlation = paired_values[column].corr(paired_values[target_column])
    sample_size = len(paired_values)
    fisher_margin = 1.959963984540054 / np.sqrt(sample_size - 3)
    confidence_interval = np.tanh(np.arctanh(correlation) + np.array([-fisher_margin, fisher_margin]))
    correlation_results.append({
        "predictor": column,
        "correlation": correlation,
        "ci_low": confidence_interval[0],
        "ci_high": confidence_interval[1],
        "sample_size": sample_size,
    })

target_correlations = (
    pd.DataFrame(correlation_results)
    .assign(significant=lambda frame: (frame["ci_low"] > 0) | (frame["ci_high"] < 0))
    .sort_values("correlation")
    .reset_index(drop=True)
)
annotation_text = target_correlations.apply(
    lambda row: f"{row['correlation']:.2f}{'*' if not row['significant'] else ''}",
    axis=1,
).to_numpy().reshape(-1, 1)

fig = go.Figure(go.Heatmap(
    z=target_correlations[["correlation"]].to_numpy(),
    x=[target_column],
    y=target_correlations["predictor"],
    colorscale="RdBu_r",
    zmin=-1,
    zmax=1,
    text=annotation_text,
    texttemplate="%{text}",
    textfont={"size": 11},
    customdata=target_correlations[["ci_low", "ci_high", "sample_size"]].to_numpy().reshape(-1, 1, 3),
    colorbar={"title": "Pearson<br>correlation"},
    hovertemplate=(
        "Predictor: %{y}<br>Correlation: %{z:.3f}<br>"
        "95% CI: [%{customdata[0]:.3f}, %{customdata[1]:.3f}]<br>"
        "n: %{customdata[2]}<extra></extra>"
    ),
))
fig.update_layout(
    title="Predictor Correlation with Churn Value (* = 95% CI includes zero)",
    xaxis_title="Target",
    yaxis_title="Predictor",
    height=max(650, 24 * len(target_correlations)),
)
fig.show()

# Insights

## Positive Correlation (More likely to churn):
- Customers on Month-to-Month contracts
- Fiber Optic Customers
- Customers who do not subscribe to a technical support plan (that provides reduced wait times)
- Customers with higher monthly charges


## Negative Correlation (Less likely to churn):
- Customers who remain longer (tenure) with the company are less likely to churn
- Customers with long-term contracts
- Customers without internet service
- Customers with dependents are less likely to churn, may indicate higher socio-economic class. Investigate the underlying needs and how our plans fit these customers.

## Additional notes:
- Interesting to see how if there is "less friction" to cancel a plan, then it is more likely to churn. However, this finding may not be actionable as to make it "more difficult" to cancel a plan, may just frustrate users more. Test value-led retention options such as flexible loyalty rewards, targeted support, or contract-upgrade incentives.
- Customers with fiber optic service show higher churn, which may indicate dissatisfaction with service quality, pricing, reliability, or expectations. This warrants further investigation using customer feedback, support tickets, outage data, and plan-level pricing

# Investigate Multi-Collinearity (using VIF)

Variance Inflation Factor (VIF) is a statistical metric used to detect and quantify the severity of multicollinearity (the extent to which independent variables in a regression model are linearly correlated with one another).

When independent variables are highly correlated, it becomes difficult for a regression model to isolate the individual effect of each variable on the dependent outcome. VIF measures how much the variance of an estimated regression coefficient is "inflated" due to this correlation with other predictors.

## How to Interpret VIF Values

Understanding your VIF numbers is straightforward using these general rules of thumb:

- VIF = 1: The variables are not correlated at all. There is no multicollinearity in your model.
- VIF between 1 and 5: There is mild to moderate correlation between the variables, but this is usually not severe enough to require action.
- VIF between 5 and 10: There is significant multicollinearity. This often warrants further investigation.
- VIF > 10: Indicates severe multicollinearity. The standard errors of the coefficients are highly inflated, leading to unstable and unreliable regression results.


In [7]:
import numpy as np
import plotly.graph_objects as go
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression

predictor_columns = [column for column in prediction_df.columns if column != target_column]

# Full one-hot groups are perfectly collinear with an intercept. Drop one reference
# category per source feature for this diagnostic only; prediction_df is unchanged.
reference_dummy_columns = []
for source_column in one_hot_features:
    dummy_columns = sorted(
        column for column in predictor_columns if column.startswith(f"{source_column}_")
    )
    reference_dummy_columns.append(dummy_columns[0])

vif_columns = [column for column in predictor_columns if column not in reference_dummy_columns]
vif_matrix = SimpleImputer(strategy="median").fit_transform(prediction_df[vif_columns])

vif_records = []
for feature_index, feature_name in enumerate(vif_columns):
    other_features = np.delete(vif_matrix, feature_index, axis=1)
    r_squared = LinearRegression().fit(other_features, vif_matrix[:, feature_index]).score(
        other_features, vif_matrix[:, feature_index]
    )
    vif = np.inf if r_squared >= 1 - 1e-12 else 1 / (1 - r_squared)
    vif_records.append({"predictor": feature_name, "r_squared": r_squared, "vif": vif})

vif_summary = pd.DataFrame(vif_records).sort_values("vif", ascending=False).reset_index(drop=True)
finite_vif = vif_summary.loc[np.isfinite(vif_summary["vif"]), "vif"]
vif_cap = max(10, float(finite_vif.max()) if not finite_vif.empty else 10)
vif_summary["vif_for_plot"] = vif_summary["vif"].replace(np.inf, vif_cap)
vif_text = vif_summary["vif"].map(lambda value: "∞" if np.isinf(value) else f"{value:.2f}").to_numpy().reshape(-1, 1)

fig = go.Figure(go.Heatmap(
    z=vif_summary[["vif_for_plot"]].to_numpy(),
    x=["VIF"],
    y=vif_summary["predictor"],
    colorscale="YlOrRd",
    zmin=1,
    zmax=vif_cap,
    text=vif_text,
    texttemplate="%{text}",
    colorbar={"title": "VIF (capped)"},
    hovertemplate="Predictor: %{y}<br>VIF: %{text}<extra></extra>",
))
fig.update_layout(
    title="Predictor Multicollinearity: Variance Inflation Factor",
    height=max(650, 24 * len(vif_summary)),
)
fig.show()

vif_summary[["predictor", "r_squared", "vif"]]

,predictor,r_squared,vif
0,Phone Service,1.000000,inf
1,Online Security_No internet service,1.000000,inf
2,Internet Service_No,1.000000,inf
3,Multiple Lines_No phone service,1.000000,inf
4,Online Backup_No internet service,1.000000,inf
5,Tech Support_No internet service,1.000000,inf
6,Streaming TV_No internet service,1.000000,inf
7,Device Protection_No internet service,1.000000,inf
8,Streaming Movies_No internet service,1.000000,inf
9,Monthly Charges,0.998844,865.318550


# Pruning Predictors due to Multicollinearity

The VIF review identified redundant representations of the same service state. The transformations below retain a single, business-readable streaming feature, remove cumulative charges that substantially overlap with tenure, and retain only one representation of customers without internet service. These changes are applied directly to `prediction_df`, so the exported modeling dataset uses the pruned predictor set.

In [8]:
# A customer is a streaming-service subscriber when they have TV, movies, or both.
prediction_df["Streaming Service"] = (
    (prediction_df["Streaming TV_Yes"] == 1)
    | (prediction_df["Streaming Movies_Yes"] == 1)
).astype(int)

columns_to_remove = [
    "Streaming TV_Yes",
    "Streaming Movies_Yes",
    "Streaming TV_No",
    "Streaming Movies_No",
    "Multiple Lines_No phone service",
    "Total Charges",
    "Total_Charges_Missing",
    "Online Security_No internet service",
    "Online Backup_No internet service",
    "Device Protection_No internet service",
    "Tech Support_No internet service",
    "Streaming TV_No internet service",
    "Streaming Movies_No internet service",
]
prediction_df = prediction_df.drop(columns=columns_to_remove)

# Logistic regression uses the pruned features and excludes Monthly Charges.
logistic_prediction_df = prediction_df.drop(columns=["Monthly Charges"]).copy()

pruned_predictor_columns = [column for column in prediction_df.columns if column != target_column]
print(f"Pruned prediction dataset: {prediction_df.shape[0]:,} rows x {prediction_df.shape[1]} columns")
print(f"Removed {len(columns_to_remove)} predictors; retained {len(pruned_predictor_columns)} predictors.")
logistic_prediction_df.columns.to_list()

Pruned prediction dataset: 7,043 rows x 28 columns
Removed 13 predictors; retained 27 predictors.


['Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Paperless Billing',
 'Churn Value',
 'Multiple Lines_No',
 'Multiple Lines_Yes',
 'Internet Service_DSL',
 'Internet Service_Fiber optic',
 'Internet Service_No',
 'Online Security_No',
 'Online Security_Yes',
 'Online Backup_No',
 'Online Backup_Yes',
 'Device Protection_No',
 'Device Protection_Yes',
 'Tech Support_No',
 'Tech Support_Yes',
 'Contract_Month-to-month',
 'Contract_One year',
 'Contract_Two year',
 'Payment Method_Bank transfer (automatic)',
 'Payment Method_Credit card (automatic)',
 'Payment Method_Electronic check',
 'Payment Method_Mailed check',
 'Streaming Service']

In [9]:
import numpy as np
import plotly.graph_objects as go
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression

# Drop one remaining dummy from each multi-column categorical group for the VIF
# regression only. The pruned prediction_df itself is not modified.
pruned_reference_dummy_columns = []
for source_column in one_hot_features:
    dummy_columns = sorted(
        column for column in pruned_predictor_columns if column.startswith(f"{source_column}_")
    )
    if len(dummy_columns) >= 2:
        pruned_reference_dummy_columns.append(dummy_columns[0])

pruned_vif_columns = [
    column for column in pruned_predictor_columns if column not in pruned_reference_dummy_columns
]
pruned_vif_matrix = SimpleImputer(strategy="median").fit_transform(prediction_df[pruned_vif_columns])

pruned_vif_records = []
for feature_index, feature_name in enumerate(pruned_vif_columns):
    other_features = np.delete(pruned_vif_matrix, feature_index, axis=1)
    r_squared = LinearRegression().fit(other_features, pruned_vif_matrix[:, feature_index]).score(
        other_features, pruned_vif_matrix[:, feature_index]
    )
    vif = np.inf if r_squared >= 1 - 1e-12 else 1 / (1 - r_squared)
    pruned_vif_records.append({"predictor": feature_name, "r_squared": r_squared, "vif": vif})

pruned_vif_summary = pd.DataFrame(pruned_vif_records).sort_values("vif", ascending=False).reset_index(drop=True)
finite_pruned_vif = pruned_vif_summary.loc[np.isfinite(pruned_vif_summary["vif"]), "vif"]
pruned_vif_cap = max(10, float(finite_pruned_vif.max()) if not finite_pruned_vif.empty else 10)
pruned_vif_summary["vif_for_plot"] = pruned_vif_summary["vif"].replace(np.inf, pruned_vif_cap)
pruned_vif_text = pruned_vif_summary["vif"].map(
    lambda value: "∞" if np.isinf(value) else f"{value:.2f}"
).to_numpy().reshape(-1, 1)

fig = go.Figure(go.Heatmap(
    z=pruned_vif_summary[["vif_for_plot"]].to_numpy(),
    x=["VIF"],
    y=pruned_vif_summary["predictor"],
    colorscale="YlOrRd",
    zmin=1,
    zmax=pruned_vif_cap,
    text=pruned_vif_text,
    texttemplate="%{text}",
    colorbar={"title": "VIF (capped)"},
    hovertemplate="Predictor: %{y}<br>VIF: %{text}<extra></extra>",
))
fig.update_layout(
    title="Pruned Predictor Multicollinearity: Variance Inflation Factor",
    height=max(650, 24 * len(pruned_vif_summary)),
)
fig.show()

pruned_vif_summary[["predictor", "r_squared", "vif"]]

,predictor,r_squared,vif
0,Monthly Charges,0.986371,73.371050
1,Internet Service_Fiber optic,0.934924,15.366732
2,Internet Service_No,0.908734,10.957008
3,Streaming Service,0.830521,5.900445
4,Phone Service,0.754467,4.072777
5,Tenure Months,0.638608,2.767077
6,Contract_Two year,0.614127,2.591525
7,Device Protection_Yes,0.534702,2.149161
8,Tech Support_Yes,0.498388,1.993572
9,Payment Method_Electronic check,0.492584,1.970770


# Export Processed Dataset

In [10]:
from pathlib import Path

project_root = Path.cwd().resolve()
if not (project_root / "data").exists():
    project_root = project_root.parent
processed_data_dir = project_root / "data/processed"
processed_data_dir.mkdir(parents=True, exist_ok=True)
logistic_csv_path = processed_data_dir / "prediction_df_logistic_regression.csv"
logistic_data_dictionary_path = processed_data_dir / "prediction_df_logistic_regression_data_dictionary.md"
xgboost_csv_path = processed_data_dir / "prediction_df_xgboost.csv"
xgboost_data_dictionary_path = processed_data_dir / "prediction_df_xgboost_data_dictionary.md"

# Make the target the final field in both exported datasets.
processed_prediction_df = logistic_prediction_df[
    [column for column in logistic_prediction_df.columns if column != target_column] + [target_column]
].copy()
xgboost_processed_prediction_df = xgboost_prediction_df[
    [column for column in xgboost_prediction_df.columns if column != target_column] + [target_column]
].copy()
processed_prediction_df.to_csv(logistic_csv_path, index=False)
xgboost_processed_prediction_df.to_csv(xgboost_csv_path, index=False)

numeric_descriptions = {
    "Tenure Months": "Number of months the customer has been with the company.",
    "Monthly Charges": "Customer's current monthly service charge.",
    "Total Charges": "Customer's cumulative service charges; blank source values are stored as missing.",
}
binary_descriptions = {
    "Partner": "Whether the customer has a partner.",
    "Dependents": "Whether the customer has dependents.",
    "Phone Service": "Whether the customer has phone service.",
    "Paperless Billing": "Whether the customer uses paperless billing.",
    "Total_Charges_Missing": "Whether Total Charges was blank or unavailable in the source data.",
    "Streaming Service": "Whether the customer subscribes to streaming TV, streaming movies, or both.",
}

def describe_processed_column(column):
    if column in numeric_descriptions:
        return "Numeric", numeric_descriptions[column], "Continuous numeric value"
    if column in binary_descriptions:
        return "Binary encoding", binary_descriptions[column], "0 = No; 1 = Yes"
    if column == target_column:
        return "Target", "Customer churn outcome.", "0 = did not churn; 1 = churned"
    for source_column in one_hot_features:
        prefix = f"{source_column}_"
        if column.startswith(prefix):
            category = column.removeprefix(prefix)
            return (
                "One-hot encoding",
                f"Whether {source_column} is '{category}'.",
                "0 = category not present; 1 = category present",
            )
    return "Processed feature", "Processed prediction feature.", "See source data documentation"

dictionary_records = []
for column in processed_prediction_df.columns:
    encoding, description, valid_values = describe_processed_column(column)
    dictionary_records.append({
        "Processed column": column,
        "Data type": str(processed_prediction_df[column].dtype),
        "Encoding": encoding,
        "Description": description,
        "Valid values": valid_values,
        "Missing values": int(processed_prediction_df[column].isna().sum()),
    })

data_dictionary = pd.DataFrame(dictionary_records)

def escape_markdown(value):
    return str(value).replace("|", "\\|").replace("\n", " ")

markdown_lines = [
    "# Logistic Regression Processed Prediction Dataset Data Dictionary",
    "",
    f"- **File:** `{logistic_csv_path.name}`",
    f"- **Rows:** {processed_prediction_df.shape[0]:,}",
    f"- **Columns:** {processed_prediction_df.shape[1]} ({processed_prediction_df.shape[1] - 1} predictors and 1 target)",
    f"- **Target (final column):** `{target_column}`",
    "",
    "## Purpose and feature preparation",
    "",
    "This dataset is the model-ready input for the logistic-regression churn classifier. It uses a multicollinearity-pruned feature set to make linear-model coefficients more stable and interpretable.",
    "",
    "- Yes/no source features are binary encoded as `0`/`1`.",
    "- Remaining categorical features are one-hot encoded.",
    "- `Streaming Service` is `1` when a customer has streaming TV, streaming movies, or both.",
    "- `Monthly Charges`, `Total Charges`, the total-charges missingness indicator, redundant streaming fields, and redundant no-service fields are excluded.",
    "- `Churn Value` is the final column and is the binary target (`1` = churned).",
    "",
    "| Processed column | Data type | Encoding | Description | Valid values | Missing values |",
    "|---|---|---|---|---|---:|",
]
for record in dictionary_records:
    markdown_lines.append(
        "| " + " | ".join(escape_markdown(record[field]) for field in data_dictionary.columns) + " |"
    )
logistic_data_dictionary_path.write_text("\n".join(markdown_lines) + "\n", encoding="utf-8")

xgboost_dictionary_records = []
for column in xgboost_processed_prediction_df.columns:
    encoding, description, valid_values = describe_processed_column(column)
    xgboost_dictionary_records.append({
        "Processed column": column,
        "Data type": str(xgboost_processed_prediction_df[column].dtype),
        "Encoding": encoding,
        "Description": description,
        "Valid values": valid_values,
        "Missing values": int(xgboost_processed_prediction_df[column].isna().sum()),
    })
xgboost_data_dictionary = pd.DataFrame(xgboost_dictionary_records)
xgboost_markdown_lines = [
    "# XGBoost Processed Prediction Dataset Data Dictionary",
    "",
    f"- **File:** `{xgboost_csv_path.name}`",
    f"- **Rows:** {xgboost_processed_prediction_df.shape[0]:,}",
    f"- **Columns:** {xgboost_processed_prediction_df.shape[1]} ({xgboost_processed_prediction_df.shape[1] - 1} predictors and 1 target)",
    f"- **Target (final column):** `{target_column}`",
    "",
    "## Purpose and feature preparation",
    "",
    "This dataset is the model-ready input for the XGBoost churn classifier. It retains the complete encoded predictor set, including correlated service and pricing fields, because tree-based XGBoost is not sensitive to linear multicollinearity in the way logistic regression is.",
    "",
    "- Yes/no source features are binary encoded as `0`/`1`.",
    "- Categorical source features with 3–5 levels are one-hot encoded.",
    "- `Total Charges` is numeric; its 11 blank source values are stored as missing (`NaN`).",
    "- `Total_Charges_Missing` is `1` when the original total-charge value was blank and `0` otherwise.",
    "- `Churn Value` is the final column and is the binary target (`1` = churned).",
    "",
    "| Processed column | Data type | Encoding | Description | Valid values | Missing values |",
    "|---|---|---|---|---|---:|",
]
for record in xgboost_dictionary_records:
    xgboost_markdown_lines.append(
        "| " + " | ".join(escape_markdown(record[field]) for field in xgboost_data_dictionary.columns) + " |"
    )
xgboost_data_dictionary_path.write_text("\n".join(xgboost_markdown_lines) + "\n", encoding="utf-8")

print(f"Exported logistic-regression data to: {logistic_csv_path}")
print(f"Created logistic data dictionary: {logistic_data_dictionary_path}")
print(f"Exported XGBoost data to: {xgboost_csv_path}")
print(f"Created XGBoost data dictionary: {xgboost_data_dictionary_path}")
data_dictionary

Exported logistic-regression data to: W:\Workstation ExtDrive\007 Data Science\001 Data Science Training\2026_016 ML Churn Model End to End\data\processed\prediction_df_logistic_regression.csv
Created logistic data dictionary: W:\Workstation ExtDrive\007 Data Science\001 Data Science Training\2026_016 ML Churn Model End to End\data\processed\prediction_df_logistic_regression_data_dictionary.md
Exported XGBoost data to: W:\Workstation ExtDrive\007 Data Science\001 Data Science Training\2026_016 ML Churn Model End to End\data\processed\prediction_df_xgboost.csv
Created XGBoost data dictionary: W:\Workstation ExtDrive\007 Data Science\001 Data Science Training\2026_016 ML Churn Model End to End\data\processed\prediction_df_xgboost_data_dictionary.md


,Processed column,Data type,Encoding,Description,Valid values,Missing values
0,Partner,Int64,Binary encoding,Whether the customer has a partner.,0 = No; 1 = Yes,0
1,Dependents,Int64,Binary encoding,Whether the customer has dependents.,0 = No; 1 = Yes,0
2,Tenure Months,int64,Numeric,Number of months the customer has been with th...,Continuous numeric value,0
3,Phone Service,Int64,Binary encoding,Whether the customer has phone service.,0 = No; 1 = Yes,0
4,Paperless Billing,Int64,Binary encoding,Whether the customer uses paperless billing.,0 = No; 1 = Yes,0
5,Multiple Lines_No,int64,One-hot encoding,Whether Multiple Lines is 'No'.,0 = category not present; 1 = category present,0
6,Multiple Lines_Yes,int64,One-hot encoding,Whether Multiple Lines is 'Yes'.,0 = category not present; 1 = category present,0
7,Internet Service_DSL,int64,One-hot encoding,Whether Internet Service is 'DSL'.,0 = category not present; 1 = category present,0
8,Internet Service_Fiber optic,int64,One-hot encoding,Whether Internet Service is 'Fiber optic'.,0 = category not present; 1 = category present,0
9,Internet Service_No,int64,One-hot encoding,Whether Internet Service is 'No'.,0 = category not present; 1 = category present,0
